In [1]:
import numpy as np
import pandas as pd
import kagglehub
import os
import shutil

/home/vanishhhed/.pyenv/versions/big_data/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
data_path = "../data" #XXX should be replaced by "data"
if os.path.exists(data_path):
    shutil.rmtree(data_path)

os.makedirs(data_path, exist_ok=True)


path = kagglehub.dataset_download("patrickzel/flight-delay-and-cancellation-dataset-2019-2023",
                                  force_download=True,
                                  output_dir=data_path)

if os.path.exists(f"{data_path}/.complete"):
    shutil.rmtree(f"{data_path}/.complete")
    
if os.path.exists(f"{data_path}/dictionary.html"):
    os.remove(f"{data_path}/dictionary.html")

100%|██████████| 140M/140M [00:07<00:00, 18.6MB/s] 

Extracting files...


In [12]:
raw_df = pd.read_csv("../data/flights_sample_3m.csv")

In [4]:
raw_df.columns

Index(['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF',
       'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT'],
      dtype='str')

In [18]:
pd.set_option('display.max_columns', None)
raw_df.head()

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",11:55:00,11:51:00,-4.0,19.0,12:10:00,14:43:00,4.0,15:01:00,14:47:00,-14.0,0.0,NaN,0.0,186.0,176.0,153.0,1713.95,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",21:20:00,21:14:00,-6.0,9.0,21:23:00,22:32:00,38.0,23:15:00,23:10:00,-5.0,0.0,NaN,0.0,235.0,236.0,189.0,2251.47,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",09:54:00,10:00:00,6.0,20.0,10:20:00,12:47:00,5.0,12:52:00,12:52:00,0.0,0.0,NaN,0.0,118.0,112.0,87.0,1094.35,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",16:09:00,16:08:00,-1.0,27.0,16:35:00,18:44:00,9.0,18:29:00,18:53:00,24.0,0.0,NaN,0.0,260.0,285.0,249.0,2557.24,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",18:40:00,18:38:00,-2.0,15.0,18:53:00,20:26:00,14.0,20:41:00,20:40:00,-1.0,0.0,NaN,0.0,181.0,182.0,153.0,1585.20,NaN,NaN,NaN,NaN,NaN


In [11]:

times = raw_df[['DEP_TIME', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'ARR_TIME', "WHEELS_ON", "WHEELS_OFF"]]
times.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 6 columns):
 #   Column        Dtype  
---  ------        -----  
 0   DEP_TIME      float64
 1   CRS_DEP_TIME  int64  
 2   CRS_ARR_TIME  int64  
 3   ARR_TIME      float64
 4   WHEELS_ON     float64
 5   WHEELS_OFF    float64
dtypes: float64(4), int64(2)
memory usage: 137.3 MB


In [3]:
def convert_hhmm_to_time(series: pd.Series) -> pd.Series:

    # Convert to int, NaN → None
    cleaned = series.replace('', np.nan).astype('Int64')  # Int64 поддерживает NaN
    
    def _to_time(x):
        if pd.isna(x):
            return None
        x = int(x)
        hh = x // 100
        mm = x % 100

        if hh > 23 or mm > 59:
            return None
        return f"{hh:02d}:{mm:02d}:00"
    
    return cleaned.apply(_to_time)

In [4]:
time_columns = ['DEP_TIME', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'ARR_TIME', "WHEELS_ON", "WHEELS_OFF"]

for col in time_columns:
    raw_df[col] = convert_hhmm_to_time(raw_df[col])

In [15]:
times = raw_df[['DEP_TIME', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'ARR_TIME', "WHEELS_ON", "WHEELS_OFF"]]
times.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 6 columns):
 #   Column        Dtype
---  ------        -----
 0   DEP_TIME      str  
 1   CRS_DEP_TIME  str  
 2   CRS_ARR_TIME  str  
 3   ARR_TIME      str  
 4   WHEELS_ON     str  
 5   WHEELS_OFF    str  
dtypes: str(6)
memory usage: 137.3 MB


In [5]:
def convert_miles_to_km(series: pd.Series, round_to: int = 2) -> pd.Series:
    # 1 mile = 1.60934 km
    km = series.astype(float) * 1.60934
    return km.round(round_to)

raw_df['DISTANCE'] = convert_miles_to_km(raw_df['DISTANCE'])

In [ ]:
subset_cols=['FL_DATE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME']
duplicated_groups = raw_df.groupby(subset_cols).filter(lambda x: len(x) > 1)
duplicated_groups

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
28005,2020-04-07,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,470,DFW,"Dallas/Fort Worth, TX",FLL,"Fort Lauderdale, FL",...,0.0,138.0,NaN,NaN,1800.85,NaN,NaN,NaN,NaN,NaN
2061156,2020-04-07,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,470,DFW,"Dallas/Fort Worth, TX",MCO,"Orlando, FL",...,0.0,161.0,140.0,115.0,1585.20,NaN,NaN,NaN,NaN,NaN


In [8]:
print("=== 1. БАЗОВЫЕ ПРОВЕРКИ ===")
print(f"Размер датасета: {raw_df.shape[0]:,} строк × {raw_df.shape[1]} колонок")
print(f"Память: {raw_df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# Пропущенные значения
missing = raw_df.isnull().mean() * 100
print("\nПропущенные значения (%):")
print(missing[missing > 0].sort_values(ascending=False).round(2))

# Типы данных
print("\nТипы данных:")
print(raw_df.dtypes)

=== 1. БАЗОВЫЕ ПРОВЕРКИ ===
Размер датасета: 3,000,000 строк × 32 колонок
Память: 2.76 GB

Пропущенные значения (%):
CANCELLATION_CODE          97.36
DELAY_DUE_NAS              82.20
DELAY_DUE_WEATHER          82.20
DELAY_DUE_CARRIER          82.20
DELAY_DUE_SECURITY         82.20
DELAY_DUE_LATE_AIRCRAFT    82.20
AIR_TIME                    2.87
ARR_DELAY                   2.87
ELAPSED_TIME                2.87
ARR_TIME                    2.71
WHEELS_ON                   2.70
TAXI_IN                     2.66
WHEELS_OFF                  2.64
TAXI_OUT                    2.63
DEP_TIME                    2.60
DEP_DELAY                   2.59
CRS_ARR_TIME                0.00
CRS_ELAPSED_TIME            0.00
dtype: float64

Типы данных:
FL_DATE                        str
AIRLINE                        str
AIRLINE_DOT                    str
AIRLINE_CODE                   str
DOT_CODE                     int64
FL_NUMBER                    int64
ORIGIN                         str
ORIGIN_CITY    

In [9]:
subset_key = ['FL_DATE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME']

dupe_count = raw_df.duplicated(subset=subset_key).sum()
print(f"\nДубликатов по естественному ключу: {dupe_count}")

if dupe_count > 0:
    print("→ Будем удалять дубликаты перед загрузкой!")


Дубликатов по естественному ключу: 0


In [33]:
print("\n=== 3. ПРОВЕРКИ ПО БИЗНЕС-ЛОГИКЕ ===")

checks = {
    "DEP_DELAY / ARR_DELAY": lambda df: ((df['DEP_DELAY'] < -1440) | (df['DEP_DELAY'] > 1440)).sum(),
    "DISTANCE (км)":         lambda df: ((df['DISTANCE'] <= 0) | (df['DISTANCE'] > 20000)).sum(),
    "CANCELLED / DIVERTED":  lambda df: ((~df['CANCELLED'].isin([0, 1, True, False])) | 
                                         (~df['DIVERTED'].isin([0, 1, True, False]))).sum(),
    "CANCELLATION_CODE":     lambda df: df['CANCELLATION_CODE'].isin(['A','B','C','D']).sum() 
                                        if 'CANCELLATION_CODE' in df else 0,
    "Время вылета после конвертации": lambda df: (~df['CRS_DEP_TIME'].astype(str).str.match(r'^\d{2}:\d{2}:00$')).sum()
}

for name, func in checks.items():
    bad = func(raw_df)
    print(f"{name:35} → {bad:6,} плохих значений")


=== 3. ПРОВЕРКИ ПО БИЗНЕС-ЛОГИКЕ ===
DEP_DELAY / ARR_DELAY               →    131 плохих значений
DISTANCE (км)                       →      0 плохих значений
CANCELLED / DIVERTED                →      0 плохих значений
CANCELLATION_CODE                   → 79,140 плохих значений
Время вылета после конвертации      →      0 плохих значений


In [12]:
print("\n=== 4. ЛОГИЧЕСКИЕ ПРОВЕРКИ ===")

# 1. Если рейс отменён → задержки должны быть NaN
cancel_bad = raw_df[raw_df['CANCELLED'] == 1]['ARR_DELAY'].notna().sum()
print(f"Отменённые рейсы с заполненной ARR_DELAY: {cancel_bad}")

# 2. Сумма причин задержки ≈ ARR_DELAY (с допуском)
delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
              'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']

raw_df['delay_sum'] = raw_df[delay_cols].sum(axis=1, skipna=True)
diff = (raw_df['delay_sum'] - raw_df['ARR_DELAY']).abs()
inconsistent = (diff > 5).sum()   # допуск ±5 минут
print(f"Несогласованность причин задержки (>5 мин): {inconsistent}")


=== 4. ЛОГИЧЕСКИЕ ПРОВЕРКИ ===
Отменённые рейсы с заполненной ARR_DELAY: 0
Несогласованность причин задержки (>5 мин): 1792565


In [18]:
raw_df[['AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE']].value_counts()

AIRLINE                             AIRLINE_DOT                             AIRLINE_CODE  DOT_CODE
Southwest Airlines Co.              Southwest Airlines Co.: WN              WN            19393       576470
Delta Air Lines Inc.                Delta Air Lines Inc.: DL                DL            19790       395239
American Airlines Inc.              American Airlines Inc.: AA              AA            19805       383106
SkyWest Airlines Inc.               SkyWest Airlines Inc.: OO               OO            20304       343737
United Air Lines Inc.               United Air Lines Inc.: UA               UA            19977       254504
Republic Airline                    Republic Airline: YX                    YX            20452       143107
Envoy Air                           Envoy Air: MQ                           MQ            20398       121256
JetBlue Airways                     JetBlue Airways: B6                     B6            20409       112844
Endeavor Air Inc.            

In [23]:
airplanes_df = raw_df[['AIRLINE_CODE', 'AIRLINE', 'AIRLINE_DOT', 'DOT_CODE']].rename(columns={
    'AIRLINE_CODE': 'airline_code', 
    'AIRLINE': 'airline_name', 
    'AIRLINE_DOT': 'airline_dot',
    'DOT_CODE': 'dot_code'
})
airplanes_df.drop_duplicates(inplace=True)
airplanes_df.reset_index(inplace=True, drop=True)
airplanes_df

,airline_code,airline_name,airline_dot,dot_code
0,UA,United Air Lines Inc.,United Air Lines Inc.: UA,19977
1,DL,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,19790
2,NK,Spirit Air Lines,Spirit Air Lines: NK,20416
3,WN,Southwest Airlines Co.,Southwest Airlines Co.: WN,19393
4,AA,American Airlines Inc.,American Airlines Inc.: AA,19805
5,YX,Republic Airline,Republic Airline: YX,20452
6,AS,Alaska Airlines Inc.,Alaska Airlines Inc.: AS,19930
7,B6,JetBlue Airways,JetBlue Airways: B6,20409
8,OH,PSA Airlines Inc.,PSA Airlines Inc.: OH,20397
9,G4,Allegiant Air,Allegiant Air: G4,20368


In [32]:
airports_origin = raw_df[['ORIGIN', 'ORIGIN_CITY']].rename(columns={
    'ORIGIN': 'iata_code', 
    'ORIGIN_CITY': 'city_name'
})
airports_dest = raw_df[['DEST', 'DEST_CITY']].rename(columns={
    'DEST': 'iata_code', 
    'DEST_CITY': 'city_name'
})
airports = pd.concat([airports_origin, airports_dest], ignore_index=True)
airports.drop_duplicates(inplace=True)
airports.reset_index(inplace=True, drop=True)
print(airports.shape)
airports

(381, 2)


,iata_code,city_name
0,FLL,"Fort Lauderdale, FL"
1,MSP,"Minneapolis, MN"
2,DEN,"Denver, CO"
3,MCO,"Orlando, FL"
4,DAL,"Dallas, TX"
...,...,...
376,OGD,"Ogden, UT"
377,STC,"St. Cloud, MN"
378,UIN,"Quincy, IL"
379,GST,"Gustavus, AK"


In [28]:
print(airports_origin.shape)
print(airports_dest.shape)

(3000000, 2)
(3000000, 2)


In [ ]:
airports_origin.drop_duplicates(inplace=True)
airports_dest.drop_duplicates(inplace=True)

airports.con

(381, 2)
(381, 2)


In [34]:
delay_due_df = raw_df[['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']]

In [39]:
delay_due_df.dropna(inplace=True)
delay_due_df.reset_index(inplace=True, drop=True)
delay_due_df

,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,0.0,0.0,24.0,0.0,0.0
1,141.0,0.0,0.0,0.0,0.0
2,0.0,0.0,23.0,0.0,0.0
3,60.0,0.0,0.0,0.0,0.0
4,0.0,0.0,35.0,0.0,0.0
...,...,...,...,...,...
533858,201.0,0.0,110.0,0.0,0.0
533859,0.0,0.0,16.0,0.0,0.0
533860,0.0,0.0,1.0,0.0,20.0
533861,206.0,0.0,0.0,0.0,8.0


In [13]:
print(raw_df.columns)
flights_df = raw_df.drop(columns=['AIRLINE', 'AIRLINE_DOT', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN_CITY', 'DEST_CITY'])

print(f"\n{flights_df.columns}")

flights_df.rename(columns={
        'AIRLINE_CODE': 'airline_code',
        'ORIGIN': 'origin',
        'DEST': 'dest',
        
        'FL_DATE': 'fl_date',
        'CRS_DEP_TIME': 'crs_dep_time',
        'DEP_TIME': 'dep_time',
        'CRS_ARR_TIME': 'crs_arr_time',
        'ARR_TIME': 'arr_time',
        
        'DEP_DELAY': 'dep_delay',
        'ARR_DELAY': 'arr_delay',
        'TAXI_OUT': 'taxi_out',
        'WHEELS_OFF': 'wheels_off',
        'WHEELS_ON': 'wheels_on',
        'TAXI_IN': 'taxi_in',
        
        'CANCELLED': 'cancelled',
        'CANCELLATION_CODE': 'cancellation_code',
        'DIVERTED': 'diverted',
        
        'CRS_ELAPSED_TIME': 'crs_elapsed_time',
        'ELAPSED_TIME': 'elapsed_time',
        'AIR_TIME': 'air_time',
        'DISTANCE': 'distance',
       
        'DELAY_DUE_CARRIER': 'delay_due_carrier',
        'DELAY_DUE_WEATHER': 'delay_due_weather', 
        'DELAY_DUE_NAS': 'delay_due_nas', 
        'DELAY_DUE_SECURITY': 'delay_due_security',
        'DELAY_DUE_LATE_AIRCRAFT': 'delay_due_late_aircraft'
    }, inplace=True)

print(flights_df.columns)

Index(['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF',
       'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT'],
      dtype='str')

Index(['FL_DATE', 'AIRLINE_CODE', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME',
       'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN',
       'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED',
       'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME',
       'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
       'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT'],
      dtype='str')


In [14]:
flights_df.shape

(3000000, 26)

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

# airlines = pd.read_csv("../output/airlines.csv")
# airports = pd.read_csv("../output/airports.csv")
flights = pd.read_csv("../output/flights.csv")
# print(airports.shape)
# airports
row_orig = flights.iloc[14]

In [2]:
print(type(row_orig["dep_time"]), row_orig["dep_time"])
print(type(row_orig["crs_dep_time"]))
print(type(row_orig["crs_arr_time"]))
print(type(row_orig["arr_time"]))
print(type(row_orig["crs_elapsed_time"]))
print(type(row_orig["elapsed_time"]))
print(type(row_orig["delay_due_carrier"]))

<class 'float'> nan
<class 'str'>
<class 'str'>
<class 'float'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>


In [11]:
print(float('nan'))

nan


In [3]:
# numeric_cols = ['DEP_DELAY', 'ARR_DELAY', 'TAXI_OUT', 'TAXI_IN', 
#                     'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
#                     'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
#                     'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']

# for col in numeric_cols:
#     if col in flights.columns:
#         flights[col] = pd.to_numeric(flights[col], errors='coerce')   # приводит к float64 + NaN
#         print(f"   {col:25} → float64 with np.nan")
import numpy as np
# Для строковых колонок (время и cancellation_code) - заменяем None/NaN на np.nan
string_cols = ['dep_time', 'arr_time', 'wheels_off', 'wheels_on', 'cancellation_code']

for col in string_cols:
    if col in flights.columns:
        # Заменяем Python None и pandas NA на np.nan (float64)
        flights[col] = flights[col].apply(lambda x: np.float64(np.nan) if type(x) == type(float('nan')) else x)

In [36]:
flights.to_csv("../output/flights_2.csv", na_rep=None, encoding='utf-8', index=False)

In [14]:
flights.head(20)



,airline_code,origin,dest,fl_number,fl_date,crs_dep_time,dep_time,crs_arr_time,arr_time,dep_delay,arr_delay,taxi_out,wheels_off,wheels_on,taxi_in,cancelled,cancellation_code,diverted,crs_elapsed_time,elapsed_time,air_time,distance,delay_due_carrier,delay_due_weather,delay_due_nas,delay_due_security,delay_due_late_aircraft
0,UA,FLL,EWR,1562,2019-01-09,11:55:00,11:51:00,15:01:00,14:47:00,-4.0,-14.0,19.0,12:10:00,14:43:00,4.0,False,NaN,False,186.0,176.0,153.0,1713.95,NaN,NaN,NaN,NaN,NaN
1,DL,MSP,SEA,1149,2022-11-19,21:20:00,21:14:00,23:15:00,23:10:00,-6.0,-5.0,9.0,21:23:00,22:32:00,38.0,False,NaN,False,235.0,236.0,189.0,2251.47,NaN,NaN,NaN,NaN,NaN
2,UA,DEN,MSP,459,2022-07-22,09:54:00,10:00:00,12:52:00,12:52:00,6.0,0.0,20.0,10:20:00,12:47:00,5.0,False,NaN,False,118.0,112.0,87.0,1094.35,NaN,NaN,NaN,NaN,NaN
3,DL,MSP,SFO,2295,2023-03-06,16:09:00,16:08:00,18:29:00,18:53:00,-1.0,24.0,27.0,16:35:00,18:44:00,9.0,False,NaN,False,260.0,285.0,249.0,2557.24,0.0,0.0,24.0,0.0,0.0
4,NK,MCO,DFW,407,2020-02-23,18:40:00,18:38:00,20:41:00,20:40:00,-2.0,-1.0,15.0,18:53:00,20:26:00,14.0,False,NaN,False,181.0,182.0,153.0,1585.20,NaN,NaN,NaN,NaN,NaN
5,WN,DAL,OKC,665,2019-07-31,10:10:00,12:37:00,11:10:00,13:31:00,147.0,141.0,15.0,12:52:00,13:28:00,3.0,False,NaN,False,60.0,54.0,36.0,291.29,141.0,0.0,0.0,0.0,0.0
6,AA,DCA,BOS,2134,2023-06-11,10:10:00,10:01:00,11:59:00,11:30:00,-9.0,-29.0,23.0,10:24:00,11:22:00,8.0,False,NaN,False,109.0,89.0,58.0,642.13,NaN,NaN,NaN,NaN,NaN
7,YX,HSV,DCA,4464,2019-07-08,16:43:00,16:37:00,19:45:00,20:08:00,-6.0,23.0,22.0,16:59:00,19:27:00,41.0,False,NaN,False,122.0,151.0,88.0,986.53,0.0,0.0,23.0,0.0,0.0
8,NK,IAH,LAX,590,2023-02-12,05:30:00,05:27:00,07:17:00,07:06:00,-3.0,-11.0,11.0,05:38:00,06:58:00,8.0,False,NaN,False,227.0,219.0,200.0,2219.28,NaN,NaN,NaN,NaN,NaN
9,AS,SEA,FAI,223,2020-08-22,21:25:00,21:16:00,23:55:00,23:56:00,-9.0,1.0,19.0,21:35:00,23:53:00,3.0,False,NaN,False,210.0,220.0,198.0,2467.12,NaN,NaN,NaN,NaN,NaN


In [4]:
row = flights.iloc[14]

In [5]:
print(type(row["dep_time"]) == type(float('nan')))
# print(type(row["crs_dep_time"]))
# print(type(row["crs_arr_time"]))
# print(type(row["arr_time"]))
# print(type(row["crs_elapsed_time"]))
# print(type(row["elapsed_time"]))

True


In [6]:
print(type(row["dep_time"]), row["dep_time"])
print(type(row["delay_due_nas"]), row["delay_due_nas"])

<class 'float'> nan
<class 'numpy.float64'> nan


In [11]:
row

airline_code                       WN
origin                            SJC
dest                              LAX
fl_number                         687
fl_date                    2020-04-07
crs_dep_time                 21:55:00
dep_time                          NaN
crs_arr_time                 23:15:00
arr_time                          NaN
dep_delay                         NaN
arr_delay                         NaN
taxi_out                          NaN
wheels_off                        NaN
wheels_on                         NaN
taxi_in                           NaN
cancelled                        True
cancellation_code                 NaN
diverted                        False
crs_elapsed_time                 80.0
elapsed_time                      NaN
air_time                          NaN
distance                       495.68
delay_due_carrier                 NaN
delay_due_weather                 NaN
delay_due_nas                     NaN
delay_due_security                NaN
delay_due_la

In [ ]:
flights = flights.astype(object).replace({np.nan: None})
flights.head(20)

,airline_code,origin,dest,fl_number,fl_date,crs_dep_time,dep_time,crs_arr_time,arr_time,dep_delay,arr_delay,taxi_out,wheels_off,wheels_on,taxi_in,cancelled,cancellation_code,diverted,crs_elapsed_time,elapsed_time,air_time,distance,delay_due_carrier,delay_due_weather,delay_due_nas,delay_due_security,delay_due_late_aircraft
0,UA,FLL,EWR,1562,2019-01-09,11:55:00,11:51:00,15:01:00,14:47:00,-4.0,-14.0,19.0,12:10:00,14:43:00,4.0,False,NaN,False,186.0,176.0,153.0,1713.95,NaN,NaN,NaN,NaN,NaN
1,DL,MSP,SEA,1149,2022-11-19,21:20:00,21:14:00,23:15:00,23:10:00,-6.0,-5.0,9.0,21:23:00,22:32:00,38.0,False,NaN,False,235.0,236.0,189.0,2251.47,NaN,NaN,NaN,NaN,NaN
2,UA,DEN,MSP,459,2022-07-22,09:54:00,10:00:00,12:52:00,12:52:00,6.0,0.0,20.0,10:20:00,12:47:00,5.0,False,NaN,False,118.0,112.0,87.0,1094.35,NaN,NaN,NaN,NaN,NaN
3,DL,MSP,SFO,2295,2023-03-06,16:09:00,16:08:00,18:29:00,18:53:00,-1.0,24.0,27.0,16:35:00,18:44:00,9.0,False,NaN,False,260.0,285.0,249.0,2557.24,0.0,0.0,24.0,0.0,0.0
4,NK,MCO,DFW,407,2020-02-23,18:40:00,18:38:00,20:41:00,20:40:00,-2.0,-1.0,15.0,18:53:00,20:26:00,14.0,False,NaN,False,181.0,182.0,153.0,1585.20,NaN,NaN,NaN,NaN,NaN
5,WN,DAL,OKC,665,2019-07-31,10:10:00,12:37:00,11:10:00,13:31:00,147.0,141.0,15.0,12:52:00,13:28:00,3.0,False,NaN,False,60.0,54.0,36.0,291.29,141.0,0.0,0.0,0.0,0.0
6,AA,DCA,BOS,2134,2023-06-11,10:10:00,10:01:00,11:59:00,11:30:00,-9.0,-29.0,23.0,10:24:00,11:22:00,8.0,False,NaN,False,109.0,89.0,58.0,642.13,NaN,NaN,NaN,NaN,NaN
7,YX,HSV,DCA,4464,2019-07-08,16:43:00,16:37:00,19:45:00,20:08:00,-6.0,23.0,22.0,16:59:00,19:27:00,41.0,False,NaN,False,122.0,151.0,88.0,986.53,0.0,0.0,23.0,0.0,0.0
8,NK,IAH,LAX,590,2023-02-12,05:30:00,05:27:00,07:17:00,07:06:00,-3.0,-11.0,11.0,05:38:00,06:58:00,8.0,False,NaN,False,227.0,219.0,200.0,2219.28,NaN,NaN,NaN,NaN,NaN
9,AS,SEA,FAI,223,2020-08-22,21:25:00,21:16:00,23:55:00,23:56:00,-9.0,1.0,19.0,21:35:00,23:53:00,3.0,False,NaN,False,210.0,220.0,198.0,2467.12,NaN,NaN,NaN,NaN,NaN


In [ ]:
# flights["cancelled"] = flights["cancelled"].apply(lambda x: True if x == 1.0 else False)
# flights["diverted"] = flights["diverted"].apply(lambda x: True if x == 1.0 else False)
flights.head(20)

fl_date                    2019-01-10
airline_code                       EV
fl_number                        3986
origin                            CHA
dest                              ORD
crs_dep_time                 08:00:00
dep_time                          NaN
dep_delay                         NaN
taxi_out                          NaN
wheels_off                        NaN
wheels_on                         NaN
taxi_in                           NaN
crs_arr_time                 09:06:00
arr_time                          NaN
arr_delay                         NaN
cancelled                        True
cancellation_code                   C
diverted                        False
crs_elapsed_time                  NaN
elapsed_time                      NaN
air_time                          NaN
distance                       806.28
delay_due_carrier                 NaN
delay_due_weather                 NaN
delay_due_nas                     NaN
delay_due_security                NaN
delay_due_la

In [7]:
airports = airports[airports['city_name'] != 'CONCORD, NC']
print(airports.shape)
airports.reset_index(inplace=True, drop=True)
airports

(380, 2)


,iata_code,city_name
0,FLL,"Fort Lauderdale, FL"
1,MSP,"Minneapolis, MN"
2,DEN,"Denver, CO"
3,MCO,"Orlando, FL"
4,DAL,"Dallas, TX"
...,...,...
375,OGD,"Ogden, UT"
376,STC,"St. Cloud, MN"
377,UIN,"Quincy, IL"
378,GST,"Gustavus, AK"


In [1]:
import pandas as pd
import numpy as np

def _convert_hhmm_to_time(series: pd.Series) -> pd.Series:

    # Convert to int, NaN -> None
    cleaned = series.replace('', np.nan).astype('Int64')
    
    def _to_time(x):
        if pd.isna(x):
            return x
        x = int(x)
        hh = x // 100
        mm = x % 100

        if hh > 23 or mm > 59:
            return None
        return f"{hh:02d}:{mm:02d}:00"
    
    return cleaned.apply(_to_time)


s1 = pd.Series([2233, 1232, np.nan, 1422])
s2 = pd.Series([5.15, 5.6, np.nan, 3.0])

print(type(s1))
print(type(s2))
s1.describe()
s2.describe()

print(s1.iloc[2], type(s1.iloc[2]))
print(s2.iloc[2], type(s2.iloc[2]))

s1.to_csv("s1_orig.csv", na_rep=None, encoding='utf-8', index=False)
# s1.to_csv("s2_orig.csv", na_rep=None, encoding='utf-8', index=False)


<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
nan <class 'numpy.float64'>
nan <class 'numpy.float64'>


In [ ]:
s1 = _convert_hhmm_to_time(s1)

print(s1.iloc[2], type(s1.iloc[2]))
print(s2.iloc[2], type(s2.iloc[2]))
# s1.to_parquet("s1_ch.parquet", na_rep=None, index=False)


df = pd.DataFrame([s1, s2])
df.to_parquet("combined.parquet")

nan <class 'numpy.float64'>
nan <class 'numpy.float64'>


In [ ]:
import pandas as pd

flights = pd.read_csv("../output/flights.csv")

airline_code                       OO
origin                            PSC
dest                              MSP
fl_number                        3699
fl_date                    2019-04-29
crs_dep_time                 07:00:00
dep_time                     06:53:00
crs_arr_time                 12:07:00
arr_time                     11:28:00
dep_delay                        -7.0
arr_delay                       -39.0
taxi_out                         12.0
wheels_off                   07:05:00
wheels_on                    11:25:00
taxi_in                           3.0
cancelled                       False
cancellation_code                 NaN
diverted                        False
crs_elapsed_time                187.0
elapsed_time                    155.0
air_time                        140.0
distance                      2018.11
delay_due_carrier                 NaN
delay_due_weather                 NaN
delay_due_nas                     NaN
delay_due_security                NaN
delay_due_la

In [5]:
flights.iloc[27860:27880]

,airline_code,origin,dest,fl_number,fl_date,crs_dep_time,dep_time,crs_arr_time,arr_time,dep_delay,...,diverted,crs_elapsed_time,elapsed_time,air_time,distance,delay_due_carrier,delay_due_weather,delay_due_nas,delay_due_security,delay_due_late_aircraft
27860,WN,SJC,SEA,4335,2019-01-16,07:20:00,07:18:00,09:30:00,09:09:00,-2.0,...,False,130.0,111.0,95.0,1120.10,NaN,NaN,NaN,NaN,NaN
27861,DL,DTW,CMH,652,2020-02-14,22:30:00,22:28:00,23:33:00,23:21:00,-2.0,...,False,63.0,53.0,30.0,249.45,NaN,NaN,NaN,NaN,NaN
27862,OO,CMH,DTW,3766,2019-12-16,12:00:00,11:54:00,13:11:00,12:54:00,-6.0,...,False,71.0,60.0,33.0,249.45,NaN,NaN,NaN,NaN,NaN
27863,AA,DFW,TUL,267,2020-03-15,22:25:00,22:21:00,23:29:00,23:47:00,-4.0,...,False,64.0,86.0,36.0,381.41,0.0,0.0,18.0,0.0,0.0
27864,F9,LAS,AUS,2006,2019-10-15,19:03:00,18:57:00,23:53:00,23:41:00,-6.0,...,False,170.0,164.0,139.0,1754.18,NaN,NaN,NaN,NaN,NaN
27865,WN,MCI,DEN,1359,2020-01-14,06:35:00,07:43:00,07:30:00,08:39:00,68.0,...,False,115.0,116.0,94.0,857.78,68.0,0.0,1.0,0.0,0.0
27866,UA,DSM,DEN,2316,2023-05-27,06:00:00,05:59:00,06:57:00,06:43:00,-1.0,...,False,117.0,104.0,82.0,947.90,NaN,NaN,NaN,NaN,NaN
27867,OH,CLT,FNT,5167,2020-08-13,14:34:00,15:04:00,16:30:00,17:18:00,30.0,...,False,116.0,134.0,87.0,893.18,0.0,30.0,18.0,0.0,0.0
27868,DL,ATL,CAE,1498,2023-05-26,21:00:00,21:01:00,21:59:00,21:51:00,1.0,...,False,59.0,50.0,33.0,308.99,NaN,NaN,NaN,NaN,NaN
27869,OO,DFW,ROW,3278,2022-05-08,21:42:00,08:16:00,22:18:00,08:44:00,634.0,...,False,96.0,88.0,65.0,700.06,0.0,0.0,0.0,0.0,626.0
